# Logistic Regression on Diabetes Data Before and After PCA

This notebook uses the diabetes dataset to compare logistic regression performance before PCA and after PCA.

## What this notebook does

1. Load the diabetes dataset from the datasets folder.
2. Separate input features and the target column.
3. Split the data into training and test sets.
4. Standardize the features.
5. Train a baseline logistic regression model.
6. Apply PCA with 2 principal components.
7. Train logistic regression again on the PCA output.
8. Compare both model accuracies in a small table.

Why this notebook is useful:
- it follows a clean step-by-step PCA workflow
- the code is simple and documented
- markdown explains each step clearly

In [50]:
from pathlib import Path

import pandas as pd

from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler


def find_dataset_path():
    """Return the correct path to the diabetes dataset."""
    path_candidates = [
        Path.cwd() / "datasets" / "diabetes.csv",
        (Path.cwd() / ".." / ".." / ".." / "datasets" / "diabetes.csv").resolve(),
    ]

    dataset_path = next((path for path in path_candidates if path.exists()), None)

    if dataset_path is None:
        raise FileNotFoundError("Could not locate datasets/diabetes.csv from the current notebook session.")

    return dataset_path


def train_and_evaluate_model(X_train_data, X_test_data, y_train_data, y_test_data):
    """Train logistic regression and return predictions and accuracy."""
    model = LogisticRegression(max_iter=1000)
    model.fit(X_train_data, y_train_data)
    predictions = model.predict(X_test_data)
    accuracy = accuracy_score(y_test_data, predictions)
    return model, predictions, accuracy

## 1. Load the diabetes dataset

We first read the dataset and inspect its structure.

The target column is `Outcome`, and the remaining columns are used as input features.

In [54]:
# Locate and load the diabetes dataset.
dataset_path = find_dataset_path()
df = pd.read_csv(dataset_path)

print("Dataset path:", dataset_path)
print("Dataset shape:", df.shape)
df.head()

Dataset path: C:\projects\learn_ml\datasets\diabetes.csv
Dataset shape: (768, 9)


,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1


## 2. Prepare the features and scale the data

In this step we:

- separate the features and target
- split the dataset into training and test sets
- standardize the feature values before training and before PCA

Standardization is important because PCA depends on feature scale.

In [55]:
# Separate the input features and the target column.
feature_cols = [column for column in df.columns if column != "Outcome"]
X = df[feature_cols]
y = df["Outcome"]

# Split the dataset into training and test data.
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
    )

# Standardize the feature values using only the training set.
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Number of input features:", len(feature_cols))
print("Training set shape:", X_train.shape)
print("Test set shape:", X_test.shape)

Number of input features: 8
Training set shape: (614, 8)
Test set shape: (154, 8)


## 3. Train logistic regression without PCA

This is the baseline model.

We train logistic regression on the standardized original features and keep the accuracy as the reference value.

In [56]:
# Train logistic regression on the standardized original feature set.
baseline_model, baseline_predictions, baseline_accuracy = train_and_evaluate_model(
    X_train_scaled,
    X_test_scaled,
    y_train,
    y_test,
    )

print(f"Baseline accuracy without PCA: {baseline_accuracy:.4f}")

Baseline accuracy without PCA: 0.7143


## 4. Apply PCA and reduce the feature space

Now we reduce the standardized feature set to 2 principal components.

This helps show how dimensionality reduction changes model input and accuracy.

In [57]:
# Fit PCA using n principal components on the standardized training data.
pca = PCA(n_components=6)
X_train_pca = pca.fit_transform(X_train_scaled)
X_test_pca = pca.transform(X_test_scaled)

print("Explained variance ratio:", pca.explained_variance_ratio_)
print("Total variance kept:", round(pca.explained_variance_ratio_.sum(), 4))
print("PCA-transformed training shape:", X_train_pca.shape)

Explained variance ratio: [0.26073778 0.21817547 0.13091879 0.10528263 0.09731332 0.08584814]
Total variance kept: 0.8983
PCA-transformed training shape: (614, 6)


## 5. Train logistic regression after PCA

We now train the same logistic regression model on the PCA-transformed data.

This shows whether a lower-dimensional representation still keeps useful predictive information.

In [58]:
# Train logistic regression on the PCA-transformed feature set.
pca_model, pca_predictions, pca_accuracy = train_and_evaluate_model(
    X_train_pca,
    X_test_pca,
    y_train,
    y_test,
    )

print(f"Accuracy with PCA (n components): {pca_accuracy:.4f}")

Accuracy with PCA (n components): 0.7338


## 6. Compare the results

The final table below helps compare the model accuracy before PCA and after PCA in one place.

In [59]:
# Put both model results into one simple table.
comparison_df = pd.DataFrame(
    {
        "Model": ["Logistic Regression without PCA", "Logistic Regression with PCA (2 components)"],
        "Accuracy": [baseline_accuracy, pca_accuracy],
    }
)

comparison_df

,Model,Accuracy
0,Logistic Regression without PCA,0.714286
1,Logistic Regression with PCA (2 components),0.733766
